## 1. Introduction

### 1.1 Background

Thyroid disorders are among the most common endocrine diseases worldwide and can significantly affect metabolism, cardiovascular health, growth, and overall well-being. The two main forms of thyroid dysfunction are **hyperthyroidism**, caused by excessive thyroid hormone production, and **hypothyroidism**, resulting from insufficient hormone production. Accurate diagnosis often relies on a combination of patient history, clinical assessment, and laboratory tests such as Thyroid Stimulating Hormone (TSH), Triiodothyronine (T3), and Thyroxine (T4).

Recent advances in machine learning have demonstrated considerable potential for supporting healthcare decision-making by identifying complex patterns within clinical data. This project explores the application of supervised machine learning techniques to multiclass thyroid disease classification.

### 1.2 Problem Statement

Accurate differentiation between hyperthyroidism, hypothyroidism, and normal thyroid function remains challenging because these conditions often exhibit overlapping clinical characteristics. Given the availability of routinely collected clinical and laboratory data, this project investigates whether supervised machine learning models can accurately classify patients into the three thyroid disease categories and identify the most effective predictive approach.

### 1.3 Key Questions

This project seeks to answer the following questions:

1. Can machine learning accurately classify patients into **Hyperthyroidism**, **Hypothyroidism**, and **Normal thyroid function** using routinely collected clinical and laboratory data?
2. Which clinical and laboratory features are most informative for distinguishing between thyroid conditions?
3. Which supervised machine learning algorithm provides the best predictive performance for multiclass thyroid disease classification?
4. How does class imbalance influence model performance across the three thyroid classes?
5. Which thyroid conditions are most frequently misclassified, and what factors may contribute to these errors?


### 1.4 Project Objective

The primary objective of this project is to develop and evaluate multiclass machine learning models capable of classifying patients into **Hyperthyroidism**, **Hypothyroidism**, or **Normal thyroid function**. In addition, the project aims to compare the performance of multiple classification algorithms and identify the clinical and laboratory features that contribute most to accurate thyroid disease prediction.


### 1.5 Success Metrics

Model performance will be assessed primarily using the Macro F1-score, which provides a balanced evaluation across all thyroid classes. Supporting metrics will include Balanced Accuracy, class-specific Precision, Recall, F1-score, and Confusion Matrices to provide a comprehensive assessment of predictive performance. Overall Accuracy will be reported for completeness but will not be the primary criterion for model selection due to the imbalanced class distribution.

## 2. Data Understanding

 This section introduces the dataset, examines its structure, and performs an initial quality assessment to identify potential issues that may require attention during data preparation. The goal is to establish a clear understanding of the available data before conducting exploratory analysis.

### 2.1 Import Libraries

The project relies on several Python libraries for data manipulation, visualization, statistical analysis, and machine learning. Importing these libraries at the beginning of the notebook ensures that all required tools are available for the subsequent analysis.

In [2]:

# Import Libraries

# Core Libraries
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical Analysis
from scipy import stats
from scipy.stats import chi2_contingency, kruskal

# Machine Learning - Preprocessing
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif, f_classif

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Boosting Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Model Interpretation
from sklearn.inspection import permutation_importance

# =====================================================
# Notebook Settings
# =====================================================

warnings.filterwarnings("ignore")

sns.set_theme(
    style="whitegrid",
    palette="viridis",
    context="notebook"
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


### 2.2 Load Dataset

The next step is to load the thyroid disease dataset into a Pandas DataFrame for inspection and analysis.


In [3]:
# Load dataset
df = pd.read_csv("data/thyroid.csv")

# Display dataset dimensions
print(f"Dataset Shape: {df.shape}")

# Display the first five records
df.head()

Dataset Shape: (7200, 22)


,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,query_hyperthyroid,lithium,goitre,tumor,hypopituitary,psych,TSH,T3,TT4,T4U,FTI,class
0,0.7300,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0006,0.0150,0.1200,0.0820,0.1460,3
1,0.2400,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0003,0.0300,0.1430,0.1330,0.1080,3
2,0.4700,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0019,0.0240,0.1020,0.1310,0.0780,3
3,0.6400,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0009,0.0170,0.0770,0.0900,0.0850,3
4,0.2300,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0003,0.0260,0.1390,0.0900,0.1530,3


### 2.3 Dataset Source

This study uses the ANN-Thyroid dataset obtained from the UCI Machine Learning Repository, a widely used benchmark repository for machine learning research. The dataset contains patient records comprising demographic information, clinical indicators, treatment history, laboratory measurements, and thyroid diagnosis labels. The original dataset consists of separate training and testing files that have been combined into a single dataset for this analysis.

### 2.4 Dataset Overview

The following inspection provides a high-level understanding of the dataset, including its dimensions, variables, data types, and memory usage.

In [4]:
# Dataset dimensions.
print (f"Rows: {df.shape[0]}")
print (f"Columns: {df.shape[1]}")

# Column names
df.columns

Rows: 7200
Columns: 22


Index(['age', 'sex', 'on_thyroxine', 'query_on_thyroxine',
       'on_antithyroid_medication', 'sick', 'pregnant', 'thyroid_surgery',
       'I131_treatment', 'query_hypothyroid', 'query_hyperthyroid', 'lithium',
       'goitre', 'tumor', 'hypopituitary', 'psych', 'TSH', 'T3', 'TT4', 'T4U',
       'FTI', 'class'],
      dtype='str')

The dataset contains 7200 observations and 22 features.

In [5]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        7200 non-null   float64
 1   sex                        7200 non-null   int64  
 2   on_thyroxine               7200 non-null   int64  
 3   query_on_thyroxine         7200 non-null   int64  
 4   on_antithyroid_medication  7200 non-null   int64  
 5   sick                       7200 non-null   int64  
 6   pregnant                   7200 non-null   int64  
 7   thyroid_surgery            7200 non-null   int64  
 8   I131_treatment             7200 non-null   int64  
 9   query_hypothyroid          7200 non-null   int64  
 10  query_hyperthyroid         7200 non-null   int64  
 11  lithium                    7200 non-null   int64  
 12  goitre                     7200 non-null   int64  
 13  tumor                      7200 non-null   int64  
 14  hyp

The dataset contains 7,200 observations and 22 variables. It comprises 6 continuous variables and 16 integer variables, including the target variable (class). The dataset is well-structured and suitable for exploratory data analysis, preprocessing, and multiclass machine learning classification.

In [6]:
# Display five random observations
df.sample(5, random_state=42)

,age,sex,on_thyroxine,query_on_thyroxine,on_antithyroid_medication,sick,pregnant,thyroid_surgery,I131_treatment,query_hypothyroid,query_hyperthyroid,lithium,goitre,tumor,hypopituitary,psych,TSH,T3,TT4,T4U,FTI,class
3098,0.4000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0024,0.0208,0.1120,0.1250,0.0900,3
2531,0.3500,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0012,0.0270,0.1370,0.1190,0.1150,3
4071,0.4600,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0013,0.0201,0.0730,0.0770,0.0950,3
1287,0.7300,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0.0039,0.0090,0.0620,0.0540,0.1150,3
2540,0.7100,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0006,0.0208,0.1190,0.1080,0.1100,3


In [10]:
#Descriptive statistics for numerical variables.
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,7200.0000,0.5205,0.1893,0.0100,0.3700,0.5500,0.6700,0.9700
sex,7200.0000,0.3043,0.4601,0.0000,0.0000,0.0000,1.0000,1.0000
on_thyroxine,7200.0000,0.1306,0.3369,0.0000,0.0000,0.0000,0.0000,1.0000
query_on_thyroxine,7200.0000,0.0154,0.1232,0.0000,0.0000,0.0000,0.0000,1.0000
on_antithyroid_medication,7200.0000,0.0128,0.1123,0.0000,0.0000,0.0000,0.0000,1.0000
sick,7200.0000,0.0383,0.1920,0.0000,0.0000,0.0000,0.0000,1.0000
pregnant,7200.0000,0.0108,0.1035,0.0000,0.0000,0.0000,0.0000,1.0000
thyroid_surgery,7200.0000,0.0140,0.1176,0.0000,0.0000,0.0000,0.0000,1.0000
I131_treatment,7200.0000,0.0168,0.1286,0.0000,0.0000,0.0000,0.0000,1.0000
query_hypothyroid,7200.0000,0.0656,0.2475,0.0000,0.0000,0.0000,0.0000,1.0000


The binary variables have means close to 0, indicating that most clinical conditions are relatively uncommon in the dataset. The continuous variable age has a mean of 0.52 with moderate variability (SD = 0.19), while the wide ranges observed in the laboratory measurements should be further examined for potential outliers during exploratory data analysis.

### 2.5 Dataset Dictionary

The dataset contains demographic variables, treatment-related variables, clinical indicators, laboratory measurements, and a multiclass target variable. The table below provides a description of each variable included in the analysis.

In [12]:
# Data Dictionary
data_dictionary = pd.DataFrame({
    "Feature": [
        "age", "sex", "on_thyroxine", "query_on_thyroxine",
        "on_antithyroid_medication", "sick", "pregnant",
        "thyroid_surgery", "I131_treatment",
        "query_hypothyroid", "query_hyperthyroid",
        "lithium", "goitre", "tumor",
        "hypopituitary", "psych",
        "TSH", "T3", "TT4", "T4U", "FTI",
        "class"
    ],

    "Type": [
        "Numeric", "Binary", "Binary", "Binary",
        "Binary", "Binary", "Binary",
        "Binary", "Binary",
        "Binary", "Binary",
        "Binary", "Binary", "Binary",
        "Binary", "Binary",
        "Numeric", "Numeric", "Numeric", "Numeric", "Numeric",
        "Categorical"
    ],

    "Category": [
        "Demographic", "Demographic",
        "Treatment", "Treatment",
        "Treatment", "Clinical", "Clinical",
        "Treatment", "Treatment",
        "Clinical", "Clinical",
        "Medication", "Clinical", "Clinical",
        "Clinical", "Clinical",
        "Laboratory", "Laboratory", "Laboratory",
        "Laboratory", "Laboratory",
        "Target"
    ],

    "Description": [
        "Normalized patient age.",
        "Patient sex (0 = Female, 1 = Male).",
        "Whether the patient is receiving thyroxine treatment.",
        "Whether thyroxine treatment was queried.",
        "Whether the patient is receiving antithyroid medication.",
        "Whether the patient was recorded as sick.",
        "Pregnancy status.",
        "History of thyroid surgery.",
        "History of radioactive iodine (I131) treatment.",
        "Whether hypothyroidism was suspected.",
        "Whether hyperthyroidism was suspected.",
        "History of lithium medication.",
        "Presence of goitre.",
        "Presence of a thyroid tumor.",
        "Presence of hypopituitarism.",
        "Presence of a psychological condition.",
        "Normalized Thyroid Stimulating Hormone (TSH) level.",
        "Normalized Triiodothyronine (T3) level.",
        "Normalized Total Thyroxine (TT4) level.",
        "Normalized Thyroxine Uptake (T4U) level.",
        "Normalized Free Thyroxine Index (FTI).",
        "Target variable (1 = Hyperthyroid, 2 = Hypothyroid, 3 = Normal)."
    ]
})

data_dictionary

,Feature,Type,Category,Description
0,age,Numeric,Demographic,Normalized patient age.
1,sex,Binary,Demographic,"Patient sex (0 = Female, 1 = Male)."
2,on_thyroxine,Binary,Treatment,Whether the patient is receiving thyroxine tre...
3,query_on_thyroxine,Binary,Treatment,Whether thyroxine treatment was queried.
4,on_antithyroid_medication,Binary,Treatment,Whether the patient is receiving antithyroid m...
5,sick,Binary,Clinical,Whether the patient was recorded as sick.
6,pregnant,Binary,Clinical,Pregnancy status.
7,thyroid_surgery,Binary,Treatment,History of thyroid surgery.
8,I131_treatment,Binary,Treatment,History of radioactive iodine (I131) treatment.
9,query_hypothyroid,Binary,Clinical,Whether hypothyroidism was suspected.
